# Modèle principal et facteurs du score

Ce notebook compare une régression logistique à un gradient boosting, règle leurs seuils sur la validation, puis évalue une seule fois le modèle retenu sur le test final. Toutes les données viennent du socle `src.decoupage` ; le test n'intervient dans aucun choix.

La préparation suppose vraie l'hypothèse de correction des libellés de la Tanzanie. Les résultats sont conditionnels à cette hypothèse.

In [1]:
import sys

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

sys.path.insert(0, "..")
from src.decoupage import decouper, evaluer
from src.modele_principal import (
    CHEMIN_MODELE,
    VARIABLES_CATEGORIELLES,
    construire_modele,
    expliquer,
    entrainer,
    meilleur_seuil,
    predire,
)

X_ent, X_val, X_test, y_ent, y_val, y_test = decouper()
X_ent.shape, X_val.shape, X_test.shape

((16466, 10), (3529, 10), (3529, 10))

## 1. Régression logistique

Toutes les variables catégorielles, y compris `education_level`, sont encodées en one-hot. L'âge est d'abord réparti en tranches puis encodé en indicatrices ; la taille du ménage est standardisée. La régularisation `C` est comparée sur la validation.

In [2]:
resultats_logistiques = []
modeles_logistiques = {}
for C in [0.3, 1.0, 3.0, 10.0]:
    modele = construire_modele(C=C)
    modele.fit(X_ent, y_ent)
    probabilites = modele.predict_proba(X_val)[:, 1]
    seuil, f1 = meilleur_seuil(y_val, probabilites)
    modeles_logistiques[C] = (modele, probabilites, seuil)
    resultats_logistiques.append({"C": C, "seuil": seuil, "f1_macro_validation": f1})

table_logistique = pd.DataFrame(resultats_logistiques).sort_values(
    "f1_macro_validation", ascending=False
)
C_retenu = float(table_logistique.iloc[0]["C"])
modele_logistique, probabilites_logistiques, seuil_logistique = modeles_logistiques[C_retenu]
table_logistique

,C,seuil,f1_macro_validation
2,3.0,0.34,0.742701
0,0.3,0.34,0.742228
3,10.0,0.34,0.742228
1,1.0,0.34,0.741842


## 2. Gradient boosting

`HistGradientBoostingClassifier` reçoit les variables catégorielles en one-hot dense et les variables numériques telles quelles. Plusieurs réglages sont comparés sur la validation ; le seuil est optimisé séparément pour chaque réglage.

In [3]:
parametres_boosting = [
    {"learning_rate": 0.10, "min_samples_leaf": 20, "l2_regularization": 0.0},
    {"learning_rate": 0.06, "min_samples_leaf": 20, "l2_regularization": 0.0},
    {"learning_rate": 0.10, "min_samples_leaf": 40, "l2_regularization": 0.0},
    {"learning_rate": 0.10, "min_samples_leaf": 20, "l2_regularization": 1.0},
    {"learning_rate": 0.05, "min_samples_leaf": 30, "l2_regularization": 1.0},
]

def construire_boosting(parametres):
    preparation = ColumnTransformer(
        [
            (
                "categories",
                OneHotEncoder(handle_unknown="ignore", sparse_output=False),
                VARIABLES_CATEGORIELLES,
            ),
            (
                "numeriques",
                "passthrough",
                ["age_of_respondent", "household_size"],
            ),
        ],
        sparse_threshold=0,
    )
    classifieur = HistGradientBoostingClassifier(
        max_iter=250,
        max_leaf_nodes=31,
        random_state=42,
        early_stopping=True,
        **parametres,
    )
    return Pipeline([("preparation", preparation), ("modele", classifieur)])

resultats_boosting = []
modeles_boosting = []
for parametres in parametres_boosting:
    modele = construire_boosting(parametres)
    modele.fit(X_ent, y_ent)
    probabilites = modele.predict_proba(X_val)[:, 1]
    seuil, f1 = meilleur_seuil(y_val, probabilites)
    modeles_boosting.append((modele, probabilites, seuil, parametres))
    resultats_boosting.append(
        {**parametres, "seuil": seuil, "f1_macro_validation": f1}
    )

table_boosting = pd.DataFrame(resultats_boosting).sort_values(
    "f1_macro_validation", ascending=False
)
position_boosting = int(table_boosting.index[0])
modele_boosting, probabilites_boosting, seuil_boosting, parametres_retenus_boosting = (
    modeles_boosting[position_boosting]
)
table_boosting

,learning_rate,min_samples_leaf,l2_regularization,seuil,f1_macro_validation
4,0.05,30,1.0,0.31,0.745928
2,0.10,40,0.0,0.30,0.743801
0,0.10,20,0.0,0.36,0.743424
3,0.10,20,1.0,0.28,0.742622
1,0.06,20,0.0,0.31,0.742565


## 3. Comparaison sur la validation

La référence naïve prédit toujours l'absence de compte. Les seuils de chaque modèle maximisent le F1 macro sur `X_val, y_val`. Les tableaux ci-dessous comparent les F1 macro globaux, par pays et par genre ; les effectifs sont communs aux modèles.

In [4]:
reference = evaluer(X_val, y_val, np.zeros(len(y_val), dtype=int))
score_logistique = evaluer(
    X_val, y_val, probabilites_logistiques >= seuil_logistique
)
score_boosting = evaluer(
    X_val, y_val, probabilites_boosting >= seuil_boosting
)
comparaison_validation = pd.concat(
    {
        "naïf": reference["f1_macro"],
        "logistique": score_logistique["f1_macro"],
        "gradient boosting": score_boosting["f1_macro"],
    },
    axis=1,
)
comparaison_validation.loc[:, "effectif"] = reference["effectif"]
comparaison_validation = comparaison_validation[
    ["effectif", "naïf", "logistique", "gradient boosting"]
]
print(
    f"Logistique : C={C_retenu:g}, seuil={seuil_logistique:.2f}; "
    f"boosting : seuil={seuil_boosting:.2f}, "
    f"paramètres={parametres_retenus_boosting}"
)
comparaison_validation

Logistique : C=3, seuil=0.34; boosting : seuil=0.31, paramètres={'learning_rate': 0.05, 'min_samples_leaf': 30, 'l2_regularization': 1.0}


,effectif,naïf,logistique,gradient boosting
groupe,,,,
Global,3529,0.4621,0.7427,0.7459
country = Kenya,911,0.4281,0.7359,0.7356
country = Rwanda,1310,0.4696,0.7017,0.7007
country = Tanzania,993,0.4760,0.7476,0.7704
country = Uganda,315,0.4776,0.7312,0.7186
gender_of_respondent = Female,2077,0.4708,0.7312,0.7340
gender_of_respondent = Male,1452,0.4492,0.7437,0.7479


La logistique est retenue pour l'interface : elle permet de fournir des contributions par variable directement à partir de ses coefficients, sans dépendance d'explicabilité supplémentaire. Avec `C=3` et le seuil `0,34`, elle atteint **0,7427** de F1 macro sur validation. Le boosting atteint **0,7459** avec le seuil `0,31`, soit seulement `0,0032` de plus ; on privilégie ici l'explicabilité directe.

Les écarts par pays restent visibles : pour la logistique, le F1 va de `0,7017` au Rwanda à `0,7476` en Tanzanie (`0,7359` au Kenya, `0,7312` en Ouganda). Le boosting est meilleur en Tanzanie (`0,7704`) mais moins bon en Ouganda (`0,7186`). Par genre, la logistique obtient `0,7312` pour les femmes et `0,7437` pour les hommes ; le boosting donne respectivement `0,7340` et `0,7479`. La référence naïve globale est `0,4621`.

Les contributions du modèle sont des **associations**, pas des causes. Elles sont conditionnelles à la préparation commune, notamment à l'hypothèse de correction des libellés tanzaniens.

## 4. Enregistrement et vérifications d'interface

Le modèle final est réentraîné uniquement sur `X_ent, y_ent`. `entrainer` choisit un seuil par validation croisée interne ; pour l'artefact de cette expérience, ce seuil est remplacé par celui choisi sur `X_val, y_val`, conformément au découpage commun.

In [5]:
objet_modele = entrainer(X_ent, y_ent)
objet_modele["seuil"] = float(seuil_logistique)
joblib.dump(objet_modele, CHEMIN_MODELE)

profil = X_val.iloc[[0]].copy()
profil["job_type"] = "Self employed"
assert "Self employed" in X_ent["job_type"].unique()
noms_variables_encodees = objet_modele["modele"].named_steps[
    "preparation"
].get_feature_names_out()
assert list(noms_variables_encodees).count("job_type_Self employed") == 1

predictions_pays = pd.DataFrame(
    [
        {"pays": pays, "probabilite": predire(profil.assign(country=pays))}
        for pays in sorted(X_ent["country"].unique())
    ]
)
predictions_genres = pd.DataFrame(
    [
        {
            "genre": genre,
            "probabilite": predire(
                profil.assign(gender_of_respondent=genre)
            ),
        }
        for genre in sorted(X_ent["gender_of_respondent"].unique())
    ]
)
assert len(predictions_pays) == 4 and predictions_pays["probabilite"].between(0, 1).all()
assert len(predictions_genres) == 2 and predictions_genres["probabilite"].between(0, 1).all()
predictions_pays, predictions_genres

(       pays  probabilite
 0     Kenya     0.054177
 1    Rwanda     0.042206
 2  Tanzania     0.018242
 3    Uganda     0.015881,
     genre  probabilite
 0  Female     0.036183
 1    Male     0.054177)

## 5. Facteurs du score et test final

Les contributions logistiques sont calculées sur l'échelle du log-odds : pour chaque variable, on somme les coefficients des modalités actives (et, pour l'âge, de la tranche active). Les contributions positives poussent le score vers le haut et les négatives vers le bas.

Pour `job_type`, l'encodeur crée une seule colonne one-hot partagée pour `Self employed` (vérifiée ci-dessus), sans colonne spécifique au pays. Les probabilités peuvent varier quand seul le pays change, car l'effet estimé du pays change ; cela ne vient pas d'un codage différent de l'emploi.

In [6]:
expliquer(profil)

,variable,contribution
0,cellphone_access,-1.137737
1,education_level,-0.949661
2,location_type,-0.714439
3,country,0.516828
4,relationship_with_head,0.276361
5,marital_status,-0.156846
6,job_type,-0.090484
7,gender_of_respondent,-0.082562
8,age_of_respondent,0.063296
9,household_size,0.034311


Le test final est utilisé une seule fois, après la sélection du modèle et du seuil sur validation. Ne pas modifier le modèle ni le seuil à partir de ces résultats.

In [7]:
probabilites_test = objet_modele["modele"].predict_proba(
    X_test[objet_modele["variables"]]
)[:, 1]
predictions_test = (probabilites_test >= objet_modele["seuil"]).astype(int)
resultats_test = evaluer(X_test, y_test, predictions_test)
resultats_test

,effectif,f1_macro
groupe,,
Global,3529,0.7318
country = Kenya,910,0.7106
country = Rwanda,1311,0.6573
country = Tanzania,993,0.7984
country = Uganda,315,0.7449
gender_of_respondent = Female,2049,0.7142
gender_of_respondent = Male,1480,0.7373


Le F1 macro global du modèle retenu sur le test final est **0,7318**. Par pays : Kenya `0,7106`, Rwanda `0,6573`, Tanzanie `0,7984`, Ouganda `0,7449`. Par genre : femmes `0,7142`, hommes `0,7373`. Ces valeurs sont rapportées une seule fois et ne sont pas utilisées pour modifier le modèle.